# Regression Metrics: MAE vs RMSE

**REGRESSION_BINDER_AUDIT_V1**

This notebook compares common regression metrics on a held-out test set, with special attention to **Mean Absolute Error (MAE)** and **Root Mean Squared Error (RMSE)**.

Key points:

- MAE and RMSE are both in the **same units as the target**.
- MAE averages absolute residual magnitudes.
- RMSE is the square root of mean squared error, so larger residuals receive disproportionately more weight.
- For the same finite set of residuals, **RMSE is always at least as large as MAE**.
- There is no universal "good" MAE or RMSE value: scale, baseline performance, uncertainty and application costs matter.
- \(R^2\) can be **negative** on evaluation data when a model performs worse than the constant-mean reference used in the \(R^2\) definition.
- Metric choice should match the application's loss/cost structure; RMSE is not automatically "better" than MAE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import make_regression
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score,
    root_mean_squared_error,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

## 1. Fit a regression model and evaluate on held-out data

The model is trained only on the training split. All reported metrics below are computed on the untouched test split.

In [ ]:
X, y = make_regression(
    n_samples=1200,
    n_features=8,
    n_informative=6,
    noise=18.0,
    random_state=RANDOM_STATE,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)
y_pred_baseline = baseline.predict(X_test)

print(f"Training samples: {len(y_train)}")
print(f"Test samples:     {len(y_test)}")

## 2. MAE, MSE, RMSE, median absolute error and \(R^2\)

For residuals \(e_i = y_i - \hat y_i\):

\[
\mathrm{MAE} = \frac{1}{n}\sum_i |e_i|
\]

\[
\mathrm{MSE} = \frac{1}{n}\sum_i e_i^2
\]

\[
\mathrm{RMSE} = \sqrt{\mathrm{MSE}}
\]

MAE and RMSE retain the target's units; MSE is in squared target units.

`root_mean_squared_error` is used directly rather than manually taking the square root of MSE.

In [ ]:
def regression_metrics(y_true, y_hat):
    return {
        "MAE": mean_absolute_error(y_true, y_hat),
        "Median absolute error": median_absolute_error(y_true, y_hat),
        "MSE": mean_squared_error(y_true, y_hat),
        "RMSE": root_mean_squared_error(y_true, y_hat),
        "R²": r2_score(y_true, y_hat),
    }

model_metrics = regression_metrics(y_test, y_pred)
baseline_metrics = regression_metrics(y_test, y_pred_baseline)

print("Linear regression")
for name, value in model_metrics.items():
    print(f"  {name:24s} {value:10.4f}")

print("\nMean-prediction baseline")
for name, value in baseline_metrics.items():
    print(f"  {name:24s} {value:10.4f}")

assert model_metrics["RMSE"] + 1e-12 >= model_metrics["MAE"]

### How to interpret these numbers

- **Lower MAE / MSE / RMSE is better**, but the absolute number is meaningful only relative to the target scale and an appropriate benchmark.
- **MAE** corresponds to average absolute error magnitude and is less dominated by a few large residuals.
- **RMSE** gives more influence to large residuals because errors are squared before averaging.
- **Median absolute error** is even more resistant to a small number of extreme residuals.
- **\(R^2\)** compares squared-error performance with a constant reference based on the evaluation target mean. The best possible value is 1.0; 0 corresponds to that constant reference, and values below 0 are possible.

## 3. Demonstrate the effect of one extreme prediction error

To isolate the metric behaviour, we take the same test predictions and deliberately make **one** prediction much worse.

This is not meant to simulate every real outlier mechanism; it simply shows how the loss functions respond to a large residual.

In [ ]:
y_pred_extreme = y_pred.copy()

# Make one existing prediction dramatically worse.
target_scale = np.std(y_test)
y_pred_extreme[0] = y_test[0] + 12 * target_scale

before_mae = mean_absolute_error(y_test, y_pred)
before_rmse = root_mean_squared_error(y_test, y_pred)

after_mae = mean_absolute_error(y_test, y_pred_extreme)
after_rmse = root_mean_squared_error(y_test, y_pred_extreme)

print(f"{'Metric':<8} {'Original':>12} {'With one extreme error':>24} {'Increase':>12}")
print("-" * 62)
print(f"{'MAE':<8} {before_mae:12.3f} {after_mae:24.3f} {after_mae-before_mae:12.3f}")
print(f"{'RMSE':<8} {before_rmse:12.3f} {after_rmse:24.3f} {after_rmse-before_rmse:12.3f}")

The RMSE typically changes more because squaring makes a very large residual contribute much more strongly to the average.

That greater sensitivity can be desirable when large misses are genuinely much more costly. If large residuals are mostly measurement errors or rare contamination, a more robust loss/metric may be more informative. The application determines which behaviour is appropriate.

## 4. Visualise residuals before and after the extreme error

In [ ]:
residuals = y_test - y_pred
residuals_extreme = y_test - y_pred_extreme

fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

axes[0].scatter(y_pred, residuals, alpha=0.65)
axes[0].axhline(0, linestyle="--")
axes[0].set_title("Original held-out residuals")
axes[0].set_xlabel("Predicted value")
axes[0].set_ylabel("Residual (y - ŷ)")

axes[1].scatter(y_pred_extreme, residuals_extreme, alpha=0.65)
axes[1].axhline(0, linestyle="--")
axes[1].set_title("After one extreme prediction error")
axes[1].set_xlabel("Predicted value")

plt.tight_layout()
plt.show()

## 5. \(R^2\) can be negative

A negative \(R^2\) does not mean "negative variance explained" in a literal physical sense. It means the model's sum of squared residuals is worse than the constant-mean reference in the \(R^2\) formula for that evaluation set.

In [ ]:
bad_prediction = np.full_like(y_test, y_test.mean() + 5 * y_test.std(), dtype=float)

print(f"Linear model R²:       {r2_score(y_test, y_pred):.4f}")
print(f"Deliberately bad R²:   {r2_score(y_test, bad_prediction):.4f}")

## 6. MAE and RMSE should be compared on the same data and scale

Because MAE and RMSE are scale-dependent, comparing their raw values across unrelated targets or datasets can be misleading.

If comparisons across scales are required, define a normalization or a baseline-relative score **explicitly** and document the denominator/reference. There is no single universally correct normalization.

In [ ]:
mae_ratio_to_baseline = model_metrics["MAE"] / baseline_metrics["MAE"]
rmse_ratio_to_baseline = model_metrics["RMSE"] / baseline_metrics["RMSE"]

print(f"MAE / baseline MAE:   {mae_ratio_to_baseline:.3f}")
print(f"RMSE / baseline RMSE: {rmse_ratio_to_baseline:.3f}")
print("Values below 1 mean lower error than this chosen baseline.")

## Takeaways

- Use a held-out set or appropriate cross-validation for performance estimation.
- MAE and RMSE answer related but different loss questions.
- RMSE is always \(\ge\) MAE on the same residuals, and it is more responsive to large errors.
- Report metric units and compare against a relevant baseline.
- Do not label fixed MAE/RMSE values as universally "good" or "bad".
- \(R^2\) can be negative.
- Inspect residuals; a single scalar metric cannot diagnose model misspecification, heteroscedasticity, dependence or distribution shift.